# RAG Pipeline: GPT4All (Local LLM) + Gemini (Embeddings) + ChromaDB (Vector Store)

This notebook implements a Retrieval-Augmented Generation (RAG) system with the following components:

- **Embeddings**: Google **Gemini** (`text-embedding-004`) is used to convert documents and queries into vectors.
- **Vector Store**: **ChromaDB** stores the document embeddings and performs similarity search to retrieve relevant context.
- **Generation (LLM)**: **GPT4All** (a local/desktop LLM, run fully offline once the model file is downloaded) generates the final answer using the retrieved context.


## 1. Install Dependencies
Run this cell first on Colab. It installs the required packages.

In [ ]:
# Install required packages (run this cell on Google Colab before proceeding)
!pip install -q gpt4all google-generativeai chromadb langchain langchain-community pypdf

## 2. Imports

In [ ]:
import os
import textwrap
from typing import List

import google.generativeai as genai
import chromadb
from chromadb.config import Settings
from gpt4all import GPT4All

## 3. Configure Gemini API Key
Get a free Gemini API key from https://aistudio.google.com/app/apikey.

On Colab, it's recommended to store the key using the "Secrets" panel (key icon in the left sidebar) under the name `GEMINI_API_KEY`, then load it with `google.colab.userdata`.

In [ ]:
# Option A: Load from Colab Secrets (recommended)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None

# Option B: Fallback - prompt for the key manually if not found in Colab Secrets
if not GEMINI_API_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key: ")

genai.configure(api_key=GEMINI_API_KEY)
print("Gemini configured.")

## 4. Sample Knowledge Base
Replace this list with your own documents (paragraphs, PDF text, etc.) for a real use case.

In [ ]:
documents: List[str] = [
    "GPT4All is an ecosystem of open-source, locally-runnable large language models that can run on consumer-grade hardware without needing a GPU or internet connection.",
    "ChromaDB is an open-source embedding database designed to make it easy to build applications with vector search and retrieval-augmented generation (RAG).",
    "Gemini is Google's family of multimodal large language models, and it also provides embedding models such as text-embedding-004 for semantic search tasks.",
    "Retrieval-Augmented Generation (RAG) combines a retriever, which fetches relevant documents from a knowledge base, with a generator (LLM), which produces a response grounded in the retrieved context.",
    "Vector databases store high-dimensional embeddings and allow fast similarity search using metrics such as cosine similarity or Euclidean distance.",
    "Local LLMs like GPT4All offer privacy advantages since data never leaves the user's machine, unlike calling a cloud-hosted LLM API.",
]

## 5. Generate Embeddings with Gemini

In [ ]:
EMBEDDING_MODEL = "models/text-embedding-004"


def embed_text(text: str, task_type: str = "retrieval_document") -> List[float]:
    """Return the Gemini embedding vector for a piece of text."""
    result = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=text,
        task_type=task_type,
    )
    return result["embedding"]


document_embeddings = [embed_text(doc, task_type="retrieval_document") for doc in documents]
print(f"Generated {len(document_embeddings)} embeddings, each of dimension {len(document_embeddings[0])}.")

## 6. Store Embeddings in ChromaDB

In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection = chroma_client.get_or_create_collection(name="rag_knowledge_base")

collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    embeddings=document_embeddings,
    documents=documents,
)

print(f"Stored {collection.count()} documents in ChromaDB.")

## 7. Retrieval Function

In [ ]:
def retrieve_context(query: str, top_k: int = 3) -> List[str]:
    """Embed the query with Gemini and fetch the most relevant documents from ChromaDB."""
    query_embedding = embed_text(query, task_type="retrieval_query")
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    return results["documents"][0]

## 8. Load the Local GPT4All Model
The first run downloads the model file (a few GB) to the Colab runtime's local storage. Choose any model from the [GPT4All model list](https://gpt4all.io/models/models3.json).

In [ ]:
GPT4ALL_MODEL_NAME = "Meta-Llama-3-8B-Instruct.Q4_0.gguf"  # change to any GPT4All-supported model

gpt4all_model = GPT4All(GPT4ALL_MODEL_NAME)
print("GPT4All model loaded.")

## 9. RAG Prompt Construction + Generation with GPT4All

In [ ]:
RAG_PROMPT_TEMPLATE = """You are a helpful assistant. Use ONLY the context below to answer the question.
If the answer is not contained in the context, say you don't know.

Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question: str, context_chunks: List[str]) -> str:
    context = "\n\n".join(context_chunks)
    return RAG_PROMPT_TEMPLATE.format(context=context, question=question)


def rag_answer(question: str, top_k: int = 3, max_tokens: int = 256) -> str:
    context_chunks = retrieve_context(question, top_k=top_k)
    prompt = build_prompt(question, context_chunks)
    with gpt4all_model.chat_session():
        response = gpt4all_model.generate(prompt, max_tokens=max_tokens)
    return response, context_chunks

## 10. Try It Out

In [ ]:
question = "What is GPT4All and why would someone use it instead of a cloud LLM?"

answer, used_context = rag_answer(question)

print("Question:\n", textwrap.fill(question, 100))
print("\nRetrieved Context:")
for i, chunk in enumerate(used_context, 1):
    print(f"  [{i}] {chunk}")
print("\nAnswer:\n", textwrap.fill(answer, 100))

## 11. (Optional) Ask Multiple Questions

In [ ]:
questions = [
    "What does ChromaDB do?",
    "Which Gemini model is used for embeddings in this notebook?",
    "Explain RAG in one sentence.",
]

for q in questions:
    a, ctx = rag_answer(q)
    print("Q:", q)
    print("A:", textwrap.fill(a, 100))
    print("-" * 80)